# Logit Lens Analysis

Purpose: Find where in the model the failure happens layer by layer.

- At which layer does the model "commit" to the wrong action token?
- Do early layers already encode the wrong prediction, or does it happen late?
- Compare layer-by-layer predictions: success cases vs failure cases
> Key question: Is the repetition counting error introduced early or late in the network?

## 1. Preparation

In [2]:
from transformer_lens import HookedTransformer, HookedTransformerConfig
import numpy as np
import pandas as pd
import torch

import sys, os
root_path = os.path.abspath("..")
src_path = os.path.join(root_path, "src")
for path in [root_path, src_path]:
    if path not in sys.path:
        sys.path.append(root_path)

from config.config import Config
from src.data import SCANDataModule,SCANTokenizer
from src.model import Transformer

In [3]:
cfg = Config()
model = Transformer(cfg)
seeds = cfg.seeds
primary_seed = seeds[0]

model.load_state_dict(torch.load(f"../results/model_seed{primary_seed}.pt"))

<All keys matched successfully>

In [4]:
import re
import pandas as pd

def parse_failure_log(filepath):
    """
    Parses SCAN failure logs and returns a list of dictionaries 
    with columns: splitname, command, target, pred.
    """
    # Regex breakdown:
    # \[(.*?)\] -> Captures everything inside the square brackets (splitname)
    # COMMAND:.*?IN:\s*(.*?)\s*OUT: -> Captures everything between IN: and OUT: (command)
    # TARGET:\s*(.*?)\s*\| -> Captures everything between TARGET: and the next | (target)
    # PRED:\s*(.*)$ -> Captures everything after PRED: to the end of the line (pred)
    pattern = re.compile(r"\[(.*?)\]\s*COMMAND:\s*(.*?)\s*\|\s*TARGET:\s*(.*?)\s*\|\s*PRED:\s*(.*)$")
    
    parsed_rows = []
    
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue  # Skip empty lines
                
            match = pattern.search(line)
            if match:
                splitname, command, target, pred = match.groups()
                
                parsed_rows.append({
                    "splitname": splitname,
                    "command": command.strip(),
                    "target": target.strip(),
                    "pred": pred.strip()
                })
                
    return parsed_rows

In [5]:
simple_data_seed42 = parse_failure_log("../results/failure_cases/failure_cases_simple_seed42.txt")
length_data_seed42 = parse_failure_log("../results/failure_cases/failure_cases_length_seed42.txt")
addprim_jump_data_seed42 = parse_failure_log("../results/failure_cases/failure_cases_addprim_jump_seed42.txt")

all_failures = pd.DataFrame(simple_data_seed42 + length_data_seed42 + addprim_jump_data_seed42)
simple_failures_seed42 = pd.DataFrame(simple_data_seed42)
length_failures_seed42 = pd.DataFrame(length_data_seed42)
addprim_jump_failures_seed42 = pd.DataFrame(addprim_jump_data_seed42)

simple_failures_seed42.head()

,splitname,command,target,pred
0,simple,<sos> IN: turn opposite right thrice and turn ...,I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_...,I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_...
1,simple,<sos> IN: run right twice after walk right twi...,I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN...,I_TURN_RIGHT I_RUN I_TURN_RIGHT I_RUN I_TURN_R...
2,simple,<sos> IN: look around right twice and turn lef...,I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN...,I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN...
3,simple,<sos> IN: jump around left thrice and run righ...,I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_L...,I_TURN_LEFT I_RUN I_TURN_LEFT I_RUN I_TURN_LEF...
4,simple,<sos> IN: run thrice and walk opposite left OUT:,I_RUN I_RUN I_RUN I_TURN_LEFT I_TURN_LEFT I_WALK,I_RUN I_RUN I_RUN I_TURN_LEFT I_TURN_LEFT I_RU...


In [6]:
for seed in seeds:
    simple_data = parse_failure_log(f"../results/failure_cases/failure_cases_simple_seed{seed}.txt")
    length_data = parse_failure_log(f"../results/failure_cases/failure_cases_length_seed{seed}.txt")
    addprim_jump_data = parse_failure_log(f"../results/failure_cases/failure_cases_addprim_jump_seed{seed}.txt")

    globals()[f"simple_data_seed{seed}"] = simple_data
    globals()[f"length_data_seed{seed}"] = length_data
    globals()[f"addprim_jump_data_seed{seed}"] = addprim_jump_data

    globals()[f"all_failures_seed{seed}"] = pd.DataFrame(simple_data + length_data + addprim_jump_data)
    globals()[f"simple_failures_seed{seed}"] = pd.DataFrame(simple_data)
    globals()[f"length_failures_seed{seed}"] = pd.DataFrame(length_data)
    globals()[f"addprim_jump_failures_seed{seed}"] = pd.DataFrame(addprim_jump_data)

## 2. Interpret with TransformerLens

In [7]:
class LogitLensEvaluator:
    def __init__(self, tl_model, dm, cfg, max_new_tokens=128):
        """
        Initializes the evaluation engine.
        
        Args:
            tl_model: Loaded HookedTransformer instance
            dm: Initialized SCANDataModule instance
            cfg: Configuration object containing device specs
            max_new_tokens (int): Max steps to roll out generation
        """
        self.model = tl_model
        self.dm = dm
        self.cfg = cfg
        self.max_new_tokens = max_new_tokens
        self.tokenizer = dm.tokenizer
        
        # Internal storage
        self.df_failures = None
        self.generation_results = []
        self.split_name = ""

    def run_logit_lens(self, df, split_name="SCAN Split", sample_n=None, random_state=42):
        """
        Runs the dynamic logit lens loop over a provided DataFrame of failure cases.
        
        Args:
            df (pd.DataFrame): DataFrame containing 'command' and 'target' columns.
            split_name (str): Label for printing metrics later (e.g., 'Simple', 'Length').
            sample_n (int, optional): If provided, randomly samples N rows to run.
            random_state (int): Seed for reproducibility when sampling.
        """
        self.split_name = split_name
        self.generation_results = [] # Clear history from previous runs
        
        # Handle optional sampling
        if sample_n is not None and sample_n < len(df):
            df_to_run = df.sample(n=sample_n, random_state=random_state)
        else:
            df_to_run = df
            
        print(f"Running dynamic logit lens on {len(df_to_run)} examples from '{self.split_name}' split...")
        
        for _, row in df_to_run.iterrows():
            command = row["command"]
            target_str = row["target"]
            
            token_list = self.tokenizer.encode(command)
            token_ids = torch.tensor(token_list, dtype=torch.long).unsqueeze(0).to(self.cfg.device)
            target_tokens = self.tokenizer.encode(target_str) 
            
            history = []
            
            for step in range(self.max_new_tokens):
                # Truncated tokens
                token_ids_truncated = token_ids[:, -self.cfg.max_seq_len:]
                logits, cache = self.model.run_with_cache(token_ids_truncated)
                
                l1_resid = cache["blocks.0.hook_resid_post"][:, -1, :]
                l2_resid = cache["blocks.1.hook_resid_post"][:, -1, :]
                
                l1_next_token = self.model.unembed(self.model.ln_final(l1_resid)).argmax(dim=-1).item()
                l2_next_token = self.model.unembed(self.model.ln_final(l2_resid)).argmax(dim=-1).item()
                
                if step < len(target_tokens):
                    true_token_str = self.tokenizer.decode([target_tokens[step]])
                else:
                    true_token_str = "<eos>"
                    
                l1_str = self.tokenizer.decode([l1_next_token])
                l2_str = self.tokenizer.decode([l2_next_token])
                
                if l2_str == true_token_str:
                    status = "Correct"
                elif l1_str == true_token_str and l2_str != true_token_str:
                    status = "Late Failure (Layer 1 Overwrite)"
                else:
                    status = "Early Failure (Layer 0 Dissolution)"
                    
                history.append({
                    "step": step,
                    "ground_truth": true_token_str,
                    "layer_1_choice": l1_str,
                    "layer_2_choice": l2_str,
                    "step_status": status
                })
                
                next_token_tensor = torch.tensor([[l2_next_token]], dtype=torch.long).to(self.cfg.device)
                token_ids = torch.cat([token_ids, next_token_tensor], dim=-1)

                if l2_next_token == 2 or l2_next_token == getattr(self.tokenizer, 'eos_token_id', None):
                    break
                    
            self.generation_results.append({
                "command": command,
                "target": target_str,
                "generation_trajectory": history
            })
            
        return self

    def print_metrics(self):
        """
        Aggregates data and prints structural research metrics.
        """
        if not self.generation_results:
            print("No analysis history found. Run .run_logit_lens() first.")
            return
            
        first_error_statuses = []
        error_steps = []
        
        for case in self.generation_results:
            trajectory = case["generation_trajectory"]
            
            for step_data in trajectory:
                if step_data["step_status"] != "Correct":
                    first_error_statuses.append(step_data["step_status"])
                    error_steps.append(step_data["step"])
                    break
                    
        status_counts = pd.Series(first_error_statuses).value_counts()
        status_percentages = pd.Series(first_error_statuses).value_counts(normalize=True) * 100
        
        print(f"\n=================== SCAN {self.split_name.upper()} SPLIT METRICS ===================")
        for status in status_counts.index:
            print(f"{status:<38}: {status_counts[status]:>4} cases ({status_percentages[status]:.1f}%)")
        print("-" * 65)
        if error_steps:
            print(f"Average generation step where failure begins: {pd.Series(error_steps).mean():.2f}")
        print("=================================================================\n")

    def get_metrics(self) -> dict:
        """
        Aggregates data and returns structural research metrics as a dict,
        without printing. Call after run_logit_lens().
        """
        if not self.generation_results:
            raise ValueError("No analysis history found. Run .run_logit_lens() first.")

        first_error_statuses = []
        error_steps = []

        for case in self.generation_results:
            trajectory = case["generation_trajectory"]
            for step_data in trajectory:
                if step_data["step_status"] != "Correct":
                    first_error_statuses.append(step_data["step_status"])
                    error_steps.append(step_data["step"])
                    break

        status_percentages = pd.Series(first_error_statuses).value_counts(normalize=True) * 100

        return {
            "l0_dissolution": status_percentages.get(
                "Early Failure (Layer 0 Dissolution)", 0.0
            ),
            "l1_overwrite": status_percentages.get(
                "Late Failure (Layer 1 Overwrite)", 0.0
            ),
            "mean_error_step": (
                pd.Series(error_steps).mean() if error_steps else float("nan")
            ),
            "n_failure_cases": len(first_error_statuses),
        }                 

    def inspect_sample(self, idx=0, only_late_failures=False):
        """
        Prints a single command's step-by-step trajectory in a clean table layout.
        """
        dataset = self.generation_results
        if only_late_failures:
            dataset = [c for c in self.generation_results if any(s["step_status"] == "Late Failure (Layer 1 Overwrite)" for s in c["generation_trajectory"])]
            
        if idx >= len(dataset):
            print(f"Index out of bounds. Available samples: {len(dataset)}")
            return

In [8]:
hooked_cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=4,
    d_model=128,       
    d_head=32,          
    d_mlp=512,        
    d_vocab= 25,
    n_ctx=128,          
    act_fn="gelu",      
    normalization_type="LN",
) 

In [9]:
tl_model = HookedTransformer(hooked_cfg)
# Load original custom weights
custom_state_dict = torch.load(f"../results/model_seed{primary_seed}.pt")

# Build a state dict for TransformerLens
tl_state_dict = {}

# Keep track of IGNORE keys
keys_to_skip = ["tfblocks.0.attn.IGNORE", "tfblocks.1.attn.IGNORE"]

for key, weight in custom_state_dict.items():
    if key in keys_to_skip:
        continue
        
    new_key = key
    
    # Map the layer block prefix: "tfblocks.0..." -> "blocks.0..."
    if new_key.startswith("tfblocks."):
        new_key = new_key.replace("tfblocks.", "blocks.")
        
    # Map the final layer norm: "ln.w" -> "ln_final.w"
    if new_key.startswith("ln."):
        new_key = new_key.replace("ln.", "ln_final.")
        
    # Map token embeddings: "embed.token_embed.weight" -> "embed.W_E"
    if new_key == "embed.token_embed.weight":
        new_key = "embed.W_E"
        
    # Map positional embeddings: "embed.pos_emb.weight" -> "pos_embed.W_pos"
    if new_key == "embed.pos_emb.weight":
        new_key = "pos_embed.W_pos"
        
    tl_state_dict[new_key] = weight

# Handle TransformerLens internal tracking values
# TransformerLens has a couple of default buffers (like mask strings) it initializes itself.
# We set strict=False so it doesn't crash over its own missing 'mask' or 'IGNORE' keys.
missing_keys, unexpected_keys = tl_model.load_state_dict(tl_state_dict, strict=False)

# Verify the load was clean
print("Missing keys:", [k for k in missing_keys if "mask" not in k and "IGNORE" not in k])
print("Unexpected keys:", unexpected_keys)

Missing keys: []
Unexpected keys: []


In [10]:
def load_tl_model_for_seed(seed: int, hooked_cfg, checkpoint_dir: str = "../results") -> HookedTransformer:
    """
    Load a fresh HookedTransformer with weights from a specific seed's
    checkpoint, porting from the custom Transformer state dict format.
    """
    tl_model = HookedTransformer(hooked_cfg)
    custom_state_dict = torch.load(f"{checkpoint_dir}/model_seed{seed}.pt")

    tl_state_dict = {}
    keys_to_skip = ["tfblocks.0.attn.IGNORE", "tfblocks.1.attn.IGNORE"]

    for key, weight in custom_state_dict.items():
        if key in keys_to_skip:
            continue
        new_key = key
        if new_key.startswith("tfblocks."):
            new_key = new_key.replace("tfblocks.", "blocks.")
        if new_key.startswith("ln."):
            new_key = new_key.replace("ln.", "ln_final.")
        if new_key == "embed.token_embed.weight":
            new_key = "embed.W_E"
        if new_key == "embed.pos_emb.weight":
            new_key = "pos_embed.W_pos"
        tl_state_dict[new_key] = weight

    missing_keys, unexpected_keys = tl_model.load_state_dict(tl_state_dict, strict=False)
    real_missing = [k for k in missing_keys if "mask" not in k and "IGNORE" not in k]

    assert len(real_missing) == 0, f"Seed {seed}: unexpected missing keys {real_missing}"
    assert len(unexpected_keys) == 0, f"Seed {seed}: unexpected keys {unexpected_keys}"

    return tl_model

In [11]:
# Load model by seed
dm = SCANDataModule(cfg)
tokenizer = dm.tokenizer
splits = ["simple","length", "addprim_jump"]
all_results = {seed: {} for seed in seeds}

for seed in seeds:
    print(f"\n{'='*60}\nSEED {seed}\n{'='*60}")

    tl_model = load_tl_model_for_seed(seed, hooked_cfg)

    # 1. Initialize the evaluator
    evaluator = LogitLensEvaluator(tl_model, dm, cfg)

    # 2. Run Simple Split
    evaluator.run_logit_lens(
        globals()[f"simple_failures_seed{seed}"],
        split_name="Simple", sample_n=None
    )
    all_results[seed]["simple"] = evaluator.get_metrics()
    evaluator.print_metrics()

    # 3. Run Length Split
    evaluator.run_logit_lens(
        globals()[f"length_failures_seed{seed}"],
        split_name="Length", sample_n=None
    )
    all_results[seed]["length"] = evaluator.get_metrics()
    evaluator.print_metrics()

    # 4. Run Add Primitive Jump Split
    evaluator.run_logit_lens(
        globals()[f"addprim_jump_failures_seed{seed}"],
        split_name="Add Prim Jump", sample_n=None
    )
    all_results[seed]["addprim_jump"] = evaluator.get_metrics()
    evaluator.print_metrics()

print(f"\nCollected results for seeds: {list(all_results.keys())}")


SEED 42
Running dynamic logit lens on 3431 examples from 'Simple' split...

=================== SCAN SIMPLE SPLIT METRICS ===================
Early Failure (Layer 0 Dissolution)   : 2335 cases (68.1%)
Late Failure (Layer 1 Overwrite)      : 1096 cases (31.9%)
-----------------------------------------------------------------
Average generation step where failure begins: 4.88

Running dynamic logit lens on 3121 examples from 'Length' split...

=================== SCAN LENGTH SPLIT METRICS ===================
Early Failure (Layer 0 Dissolution)   : 2016 cases (64.6%)
Late Failure (Layer 1 Overwrite)      : 1104 cases (35.4%)
-----------------------------------------------------------------
Average generation step where failure begins: 7.39

Running dynamic logit lens on 6095 examples from 'Add Prim Jump' split...

=================== SCAN ADD PRIM JUMP SPLIT METRICS ===================
Early Failure (Layer 0 Dissolution)   : 3826 cases (62.8%)
Late Failure (Layer 1 Overwrite)      : 2267

In [12]:
print("=" * 70)
print("CROSS-SEED AGGREGATE RESULTS")
print("=" * 70)

summary = {}
for split in splits:
    l0_vals = [all_results[s][split]["l0_dissolution"] for s in seeds]
    l1_vals = [all_results[s][split]["l1_overwrite"] for s in seeds]
    step_vals = [all_results[s][split]["mean_error_step"] for s in seeds]

    summary[split] = {
        "l0_mean": np.mean(l0_vals), "l0_std": np.std(l0_vals),
        "l1_mean": np.mean(l1_vals), "l1_std": np.std(l1_vals),
        "step_mean": np.mean(step_vals), "step_std": np.std(step_vals),
    }

    print(f"\n{split.upper()}")
    print(f"  L0 Dissolution: {summary[split]['l0_mean']:.1f}% "
          f"± {summary[split]['l0_std']:.1f}%")
    print(f"  L1 Overwrite:   {summary[split]['l1_mean']:.1f}% "
          f"± {summary[split]['l1_std']:.1f}%")
    print(f"  Mean error step: {summary[split]['step_mean']:.2f} "
          f"± {summary[split]['step_std']:.2f}")

CROSS-SEED AGGREGATE RESULTS

SIMPLE
  L0 Dissolution: 69.9% ± 4.9%
  L1 Overwrite:   30.1% ± 4.9%
  Mean error step: 6.42 ± 1.58

LENGTH
  L0 Dissolution: 69.2% ± 6.5%
  L1 Overwrite:   30.8% ± 6.5%
  Mean error step: 11.50 ± 3.90

ADDPRIM_JUMP
  L0 Dissolution: 71.5% ± 7.1%
  L1 Overwrite:   28.5% ± 7.1%
  Mean error step: 6.39 ± 1.79


In [13]:
import json
with open("../results/logit_lens_cross_seed_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

## 3. Results & Discussion

### 3.1 Simple Split

* Early-Stage State Dissolution (Layer 0 Dissolution): **68.4 ± 0.4 %**
> The predominant failure mode operates via a breakdown in the initial half of the computational pipeline. At the specific execution step $t$, the correct next-token projection fails to materialize at the boundary of Layer 0. This implies that the model's foundational features—localized in Layer 0 attention blocks or the subsequent MLP layers—are failing to preserve structural context or state tracking over long horizons. By the time information enters the final layer, the contextual representations have already dissolved into an unrecoverable latent state.

* Late-Stage Structural Overwrite (Layer 1 Overwrite): 1,071 cases (**31.6% + 0.4%**)
> This confirms a failure of compositionality rather than a failure of representation. Layer 0 computes the correct grammatical operation, but circuits within Layer 1 introduce a destructive intervention. This is frequently driven by strong, un-contextualized bigram biases or highly localized attention sinks that override the foundational hidden states right before the final unembedding projection (`ln_final` $\rightarrow$ `unembed`).

* Mean Temporal Error Onset ($\tau_{err}$): **4.18 ± 0.16 generation steps**
> The temporal metric indicates that the model is robust during the execution of brief or highly localized command strings. The degradation of hidden states is strongly correlated with sequence length and generation depth, implicating a compounding informational entropy or a structural failure in tracking long-range autoregressive dependencies.

---

### 3.2 Length Split

* Early-Stage State Dissolution (Layer 0 Dissolution): **64.4% ± 1.3%**
> Compared to the distribution-matched splits, early-stage dissolution decreases slightly here. This suggests that Layer 0 remains relatively adept at encoding the basic compositional directives even when confronted with longer command sequences. However, more than half of the failures are still born from early-stage representation decay, signaling that the lower layers are fundamentally constrained in scale-invariant tracking when sequence contexts expand past training thresholds.

* Late-Stage Structural Overwrite (Layer 1 Overwrite): **35.6% ± 1.3**
> Notably, the Length Split exhibits the highest propensity for late-stage overwrites across all test distributions. This reveals that lower-level representations frequently calculate the correct compositional trajectory, but Layer 1 suffers severe structural out-of-distribution (OOD) breakdown. Because the final layer cannot reconcile the unprecedented geometric lengths with its rigid attention-sink mechanics, it aggressively forces a collapse into overlearned bigram boundaries or premature terminations.

* Mean Temporal Error Onset ($\tau_{err}$): **6.78 ± 0.53 generation steps**
> The drastic elevation in the temporal error onset to 6.78 ± 0.53 steps provides clear empirical evidence of the model's basic generative stamina. The system does not immediately lose its structural coherence upon facing an OOD sequence. Instead, it successfully generalizes the programmatic grammar for a considerable horizon before hitting a systemic memory ceiling, confirming that state degradation is an accumulated, step-dependent function of autoregressive rollout depth.

---

### 3.3 Add Prim Jump Split

* Early-Stage State Dissolution (Layer 0 Dissolution): **63.0% ± 0.6%**
> When integrating a newly isolated primitive component (`JUMP`), early dissolution mirrors the base error profile of the Simple Split. This indicates that the introduction of a novel structural primitive triggers an identical failure of representations in Layer 0. The lower layers face a systematic bottleneck where the primitive feature mapping fails to remain coherent when embedded inside complex modifier patterns, causing early state collapse.

* Late-Stage Structural Overwrite (Layer 1 Overwrite): **37.0% ± 0.6%**
> The late overwrite behavior persists strongly when contextualizing the primitive element. Even when Layer 0 effectively maps the new primitive into the syntax and prepares to pass the correct target token upward, Layer 1’s established attention circuits act destructively. It forcefully maps the novel composition onto historic, heavily weight-biased primitive templates (such as `WALK` or `RUN` configurations), over-riding the lower layer's accurate computations.

* Mean Temporal Error Onset ($\tau_{err}$): **4.59 ± 0.17 generation steps**
> The alignment of the temporal error onset (4.59 ± 0.17) with the Simple Split suggests that local primitive composition errors share a uniform computational overhead. The model safely navigates the syntax initialization phase, but begins to break down at roughly the same structural depth. This reinforces that primitive isolation failures are bound to the exact same tracking limitations as standard execution phrases.

## 4. Next Steps: Attention Circuit Isolation

To transition from architectural localization (layers) to discrete functional circuits, the subsequent notebook (`02_attention_patterns.ipynb`) will isolate specific mathematical components driving these two failure modes.

### 4.1 Characterizing Early-Stage Dissolution ($t < 5$)
We will map the attention pattern matrices of **Layer 0** during extended generation trajectories. 
* **Hypothesis:** The early attention heads fail to adequately distribute attention weights back to long-range context markers in the input prompt (e.g., structural modifiers such as `after`, `thrice`, or `around`), precipitating the immediate state dissolution observed at step < 5.

### 4.2 Isolating Destructive Sub-Circuits in Layer 1
We will analyze the direct logit attribution of individual attention heads in **Layer 1** for the 31-37% of cases flagged as late overwrites.
* **Hypothesis:** There exists a subset of over-generalized execution or bigram heads within the final layer that heavily prioritize the immediate local history ($t-1$), effectively blinding the final unembedding layer to the programmatic routing information passed up through Layer 1's residual stream